## Setup: Imports, Constants & Core Functions

Update `GROUND_TRUTH`, `KEY_PHRASES`, and `STOPWORDS` to match your use case before running.

In [ ]:
import re
import pandas as pd
from difflib import SequenceMatcher
from datetime import datetime

# Placeholder definitions - you may want to customize these
GROUND_TRUTH = "This is a reference conversation to compare against."
KEY_PHRASES = ["reference conversation", "compare against"]
STOPWORDS = set(["is", "a", "to", "and", "the"])

def normalize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9s]", " ", text)
    text = re.sub(r"s+", " ", text).strip()
    return text

def tokenize(text):
    return [t for t in normalize(text).split() if t and t not in STOPWORDS]

def seq_sim(a, b):
    return SequenceMatcher(None, normalize(a), normalize(b)).ratio()

def jaccard(a, b):
    sa, sb = set(tokenize(a)), set(tokenize(b))
    return len(sa & sb) / len(sa | sb) if sa and sb else 0.0

def phrase_coverage(text):
    t = normalize(text)
    return sum(1 for p in KEY_PHRASES if p in t) / len(KEY_PHRASES)

def drift_score(text, reference, prev_text=None):
    a = seq_sim(text, reference)
    b = jaccard(text, reference)
    c = phrase_coverage(text)
    d = seq_sim(text, prev_text) if prev_text else a
    score = 1 - (0.20*a + 0.20*b + 0.35*c + 0.25*d)
    return round(score, 3), {
        "ref_seq": round(a, 3),
        "ref_jaccard": round(b, 3),
        "phrase_cov": round(c, 3),
        "prev_seq": round(d, 3)
    }

def parse_turns(text):
    return [b.strip() for b in re.split(r'\n\s*\n', text) if b.strip()]

def analyze_conversation(conversation_text, reference=GROUND_TRUTH, convo_name="Conversation"):
    turns = parse_turns(conversation_text)
    rows = []
    prev = None
    for i, turn in enumerate(turns, 1):
        score, parts = drift_score(turn, reference, prev)
        rows.append({
            "Conversation": convo_name,
            "Turn": i,
            "Timestamp": datetime.now().strftime("%H:%M"),
            "DriftScore": score,
            "Aligned": score < 0.40,
            "Flagged": score >= 0.60,
            **parts,
            "Text": turn[:180] + ("..." if len(turn) > 180 else "")
        })
        prev = turn
    return pd.DataFrame(rows)

def summary_table(df):
    if df.empty:
        return pd.DataFrame()
    return pd.DataFrame([
        {
            "Conversation": name,
            "Turns": len(g),
            "AvgDrift": round(g["DriftScore"].mean(), 3),
            "FlagRate": round(g["Flagged"].mean() * 100, 1),
            "AlignedRate": round(g["Aligned"].mean() * 100, 1)
        }
        for name, g in df.groupby("Conversation")
    ])

## Analyse a Single Conversation

Paste your conversation turns below (separate each turn with a blank line).

In [ ]:
convo1 = """
Paste turn 1 here.

Paste turn 2 here.

Paste turn 3 here.
"""

In [ ]:
df1 = analyze_conversation(convo1, convo_name="Run 1")
display(df1)
display(summary_table(df1))

csv_name = "drift_report_run1.csv"
df1.to_csv(csv_name, index=False)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(df1["Turn"], df1["DriftScore"], marker="o")
plt.axhline(0.60, color="red", linestyle="--", label="Flag threshold")
plt.xlabel("Turn")
plt.ylabel("Drift score")
plt.title("Drift over turns")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import os

csv_name = "drift_report_run1.csv"

# If running in Google Colab, this will trigger a download
try:
    from google.colab import files
    files.download(csv_name)
except ImportError:
    print(f"CSV saved locally as: {csv_name}")


## Example: Analysing a Second Conversation

In [ ]:
convo2 = """
Hello, I'd like to understand how conversation drift is calculated.

Can you explain the metrics used in the drift score?

Specifically, I'm interested in the sequence similarity, Jaccard index, and phrase coverage. """

df2 = analyze_conversation(convo2, convo_name="Run 2")
display(df2)

summary_df2 = summary_table(df2)
display(summary_df2)

## Comparing Summaries of Multiple Conversations

In [ ]:
# Assuming df1 is already available from the previous run
# If df1 was not run, you would need to regenerate it or ensure the kernel state has it.

# Create summary for df1 if not already present (or get it from kernel state)
summary_df1 = summary_table(df1)

# Concatenate the summary tables
combined_summary = pd.concat([summary_df1, summary_df2])
display(combined_summary)

## Visualising Drift for Multiple Conversations

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

plt.plot(df1["Turn"], df1["DriftScore"], marker="o", color="#1f77b4", label="Run 1")
plt.plot(df2["Turn"], df2["DriftScore"], marker="s", color="#2ca02c", label="Run 2")

plt.axhline(0.60, color="red", linestyle="--", label="Flag threshold (0.60)")
plt.axhline(0.40, color="orange", linestyle=":", label="Aligned threshold (0.40)")
plt.xlabel("Turn")
plt.ylabel("Drift Score")
plt.title("Drift Score Comparison: Multiple Conversations")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
